In [1]:
def main(datasources, start_date, end_date):
    """BigAlpha 2026 H2: self-contained, causal and calendar-complete."""
    import gc
    import warnings
    from typing import Dict, Iterator, List, Optional, Tuple

    import dai
    import numpy as np
    import pandas as pd


    # ---------------------------------------------------------------------------
    # Fixed configuration
    # ---------------------------------------------------------------------------

    BASELINE_MODE = "chrf_phase_space_hysteresis_h1"
    FACTOR_CACHE_VERSION = "chrf_h2_friction_hysteresis_l020"
    ABLATION_MODE = "main"
    VALID_ABLATION_MODES = {
        "main",
        "drop_activity_crowding",
        "drop_obi_persistence",
        "drop_depth_coherence",
    }

    FORMATION_START = "14:30:00"
    FORMATION_END = "14:45:00"       # excluded
    TAIL_START = "14:45:00"
    TAIL_END = "15:00:00"            # included

    FORMATION_START_MINUTE = 14 * 60 + 30
    FORMATION_END_MINUTE = 14 * 60 + 45
    TAIL_START_MINUTE = 14 * 60 + 45
    TAIL_END_MINUTE = 15 * 60

    MIN_FORMATION_BARS = 10
    MIN_TAIL_BARS = 10
    MIN_DAY_BARS = 100
    MIN_CROSS_SECTION = 20
    MIN_DEPTH_TOTAL = 1e-8
    ROBUST_CLIP = 5.0
    GATE_SCORE_CLIP = 6.0
    EXPECTED_LATE_SHARE = 30.0 / 240.0
    WEIGHT_CLOSING = 0.6923
    WEIGHT_ACTIVITY = 0.3077
    CHUNK_DAYS = 10
    LOW_COVERAGE_WARNING = 0.60
    EPS = 1e-12
    FRICTION_LAMBDA = 0.20
    WARMUP_CALENDAR_DAYS = 92

    # Verified from raw BigAlpha data: amount is already one-minute turnover.
    AMOUNT_IS_CUMULATIVE = False

    BID_VOLUME_COLUMNS = [f"bid_volume{i}" for i in range(1, 6)]
    ASK_VOLUME_COLUMNS = [f"ask_volume{i}" for i in range(1, 6)]
    DEPTH_COLUMNS = BID_VOLUME_COLUMNS + ASK_VOLUME_COLUMNS
    REQUIRED_COLUMNS = [
        "date",
        "instrument",
        "close",
        "amount",
        *DEPTH_COLUMNS,
    ]

    # Auditable daily diagnostics from the most recent main() call.
    _LAST_CHRF_AUDIT = pd.DataFrame()


    # ---------------------------------------------------------------------------
    # General utilities
    # ---------------------------------------------------------------------------

    def set_ablation_mode(mode: str) -> None:
        """Switch a permitted audit mode. Production default is always 'main'."""
        global ABLATION_MODE
        if mode not in VALID_ABLATION_MODES:
            raise ValueError(
                f"Unknown mode {mode!r}; choose from "
                f"{sorted(VALID_ABLATION_MODES)}"
            )
        ABLATION_MODE = mode
        print(f"CHRF mode set to: {ABLATION_MODE}")


    def get_last_chrf_audit() -> pd.DataFrame:
        """Return a copy of daily diagnostics without changing main() output."""
        return _LAST_CHRF_AUDIT.copy()


    def _as_timestamp(value: object, name: str) -> pd.Timestamp:
        result = pd.Timestamp(value)
        if pd.isna(result):
            raise ValueError(f"{name} is invalid")
        if result.tzinfo is not None:
            result = result.tz_localize(None)
        return result


    def _as_frame(value: object) -> pd.DataFrame:
        if isinstance(value, pd.DataFrame):
            return value
        if hasattr(value, "df"):
            return value.df()
        raise TypeError("dai.query result is not a DataFrame and has no .df()")


    def _date_chunks(
        start: pd.Timestamp,
        end: pd.Timestamp,
    ) -> Iterator[Tuple[pd.Timestamp, pd.Timestamp]]:
        """Yield chunks whose boundaries are at midnight, never inside a day."""
        cursor = start
        while cursor <= end:
            next_boundary = cursor.normalize() + pd.Timedelta(days=CHUNK_DAYS)
            chunk_end = min(end, next_boundary - pd.Timedelta(microseconds=1))
            yield cursor, chunk_end
            cursor = next_boundary


    def robust_zscore(
        series: pd.Series,
        clip: float = ROBUST_CLIP,
    ) -> pd.Series:
        """Same-day robust cross-sectional z-score."""
        values = pd.to_numeric(series, errors="coerce").astype("float64")
        finite = np.isfinite(values)
        output = pd.Series(np.nan, index=series.index, dtype="float64")
        if finite.sum() < MIN_CROSS_SECTION:
            return output

        valid = values.loc[finite]
        median = valid.median()
        mad = (valid - median).abs().median()
        if not np.isfinite(mad) or mad <= EPS:
            # Requirement: identical cross-section becomes zero, not division by 0.
            output.loc[finite] = 0.0
            return output

        output.loc[finite] = (
            (valid - median) / (1.4826 * mad)
        ).clip(-clip, clip)
        return output


    def _cross_sectional_robust_z(
        frame: pd.DataFrame,
        source: str,
        target: str,
    ) -> pd.DataFrame:
        frame[target] = frame.groupby(
            "trade_date",
            group_keys=False,
            observed=True,
        )[source].transform(robust_zscore)
        return frame


    def validate_input_columns(frame: pd.DataFrame) -> None:
        """Fail explicitly instead of inventing unavailable order-book fields."""
        missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
        if missing:
            raise KeyError(
                "bigalpha_2026_stock_bar1m is missing required fields: "
                f"{missing}. No substitute fields will be fabricated."
            )


    def _query_minute_chunk(
        source: str,
        start: pd.Timestamp,
        end: pd.Timestamp,
    ) -> pd.DataFrame:
        columns_sql = ", ".join(REQUIRED_COLUMNS)
        sql = f"""
            SELECT {columns_sql}
            FROM {source}
            ORDER BY instrument, date
        """
        filters = {
            "date": [
                start.strftime("%Y-%m-%d %H:%M:%S.%f"),
                end.strftime("%Y-%m-%d %H:%M:%S.%f"),
            ]
        }
        frame = _as_frame(
            dai.query(sql, filters=filters, compression=True)
        )
        validate_input_columns(frame)
        return frame.loc[:, REQUIRED_COLUMNS]


    # ---------------------------------------------------------------------------
    # Minute data preparation
    # ---------------------------------------------------------------------------

    def prepare_minute_data(raw: pd.DataFrame) -> pd.DataFrame:
        """Parse, sort and validate timestamps without filling microstructure data."""
        if raw.empty:
            return raw.copy()

        frame = raw.copy()
        parsed = pd.to_datetime(frame["date"], errors="coerce")
        timezone_detected = getattr(parsed.dt, "tz", None) is not None
        if timezone_detected:
            parsed = parsed.dt.tz_localize(None)
        frame["date"] = parsed
        frame["instrument"] = frame["instrument"].astype("string")

        invalid_key = frame["date"].isna() | frame["instrument"].isna()
        duplicate_count = int(
            frame.loc[~invalid_key].duplicated(
                ["instrument", "date"], keep=False
            ).sum()
        )
        frame = frame.loc[~invalid_key].sort_values(
            ["instrument", "date"], kind="mergesort"
        )
        frame = frame.drop_duplicates(
            ["instrument", "date"], keep="last"
        )
        frame["trade_date"] = frame["date"].dt.normalize()

        numeric_columns = ["close", "amount", *DEPTH_COLUMNS]
        for column in numeric_columns:
            frame[column] = pd.to_numeric(
                frame[column], errors="coerce"
            )

        frame["invalid_close"] = (
            ~np.isfinite(frame["close"]) | (frame["close"] <= 0)
        )
        negative_depth = frame[DEPTH_COLUMNS].lt(0).any(axis=1)
        nonfinite_depth = ~np.isfinite(frame[DEPTH_COLUMNS]).all(axis=1)
        frame["invalid_depth"] = negative_depth | nonfinite_depth
        frame["duplicate_timestamp_count"] = duplicate_count
        frame["timezone_was_removed"] = bool(timezone_detected)

        clock = frame["date"].dt.hour * 60 + frame["date"].dt.minute
        formation = (
            (clock >= FORMATION_START_MINUTE)
            & (clock < FORMATION_END_MINUTE)
        )
        tail = (
            (clock >= TAIL_START_MINUTE)
            & (clock <= TAIL_END_MINUTE)
        )
        frame["window"] = np.select(
            [formation, tail],
            ["formation", "tail"],
            default="other",
        )
        return frame


    def convert_cumulative_fields(frame: pd.DataFrame) -> pd.DataFrame:
        """
        Convert cumulative amount into one-minute increments.

        A clear reset (current <= 20% of previous) starts a new cumulative segment
        and uses the current non-negative value. Other negative differences are
        marked invalid and never enter activity calculations.
        """
        result = frame
        keys = [result["instrument"], result["trade_date"]]
        amount = result["amount"].where(
            np.isfinite(result["amount"]) & (result["amount"] >= 0)
        )

        if not AMOUNT_IS_CUMULATIVE:
            result["amount_increment"] = amount.astype("float32")
            result["amount_reset"] = False
            result["amount_anomaly"] = amount.isna()
            return result

        previous = amount.groupby(keys, sort=False).shift(1)
        difference = amount - previous
        first_observation = previous.isna() & amount.notna()
        clear_reset = (
            previous.notna()
            & amount.notna()
            & (difference < 0)
            & (amount <= 0.20 * previous)
        )
        anomaly = (
            previous.notna()
            & amount.notna()
            & (difference < 0)
            & ~clear_reset
        )

        increment = difference.where(difference >= 0)
        increment = increment.where(~first_observation, amount)
        increment = increment.where(~clear_reset, amount)
        increment = increment.where(~anomaly, np.nan)

        result["amount_increment"] = increment.astype("float32")
        result["amount_reset"] = clear_reset
        result["amount_anomaly"] = anomaly | amount.isna()
        return result


    def calculate_minute_returns(frame: pd.DataFrame) -> pd.DataFrame:
        """Backward-looking minute log returns within instrument and trade date."""
        result = frame
        valid_close = result["close"].where(~result["invalid_close"])
        keys = [result["instrument"], result["trade_date"]]
        previous = valid_close.groupby(keys, sort=False).shift(1)
        minute_return = np.log(valid_close / previous)
        result["valid_close"] = valid_close
        result["minute_return"] = minute_return.astype("float32")
        result["return_squared"] = np.square(minute_return).astype("float32")
        return result


    def calculate_obi_features(frame: pd.DataFrame) -> pd.DataFrame:
        """Calculate normalized displayed-book imbalance at levels 1 and 1-5."""
        result = frame
        valid_depth = ~result["invalid_depth"]

        bid1 = result["bid_volume1"].where(valid_depth)
        ask1 = result["ask_volume1"].where(valid_depth)
        bid5 = result[BID_VOLUME_COLUMNS].sum(
            axis=1, min_count=len(BID_VOLUME_COLUMNS)
        ).where(valid_depth)
        ask5 = result[ASK_VOLUME_COLUMNS].sum(
            axis=1, min_count=len(ASK_VOLUME_COLUMNS)
        ).where(valid_depth)

        total1 = bid1 + ask1
        total5 = bid5 + ask5
        result["obi1"] = (
            (bid1 - ask1) / total1
        ).where(total1 > MIN_DEPTH_TOTAL).clip(-1, 1).astype("float32")
        result["obi5"] = (
            (bid5 - ask5) / total5
        ).where(total5 > MIN_DEPTH_TOTAL).clip(-1, 1).astype("float32")
        result["abs_obi5"] = result["obi5"].abs().astype("float32")
        return result


    # ---------------------------------------------------------------------------
    # Daily CHRF modules
    # ---------------------------------------------------------------------------

    def _window_statistics(
        minute: pd.DataFrame,
        window_name: str,
    ) -> pd.DataFrame:
        keys = ["trade_date", "instrument"]
        selected = minute.loc[minute["window"].eq(window_name)]
        if selected.empty:
            return pd.DataFrame()

        grouped = selected.groupby(keys, sort=False, observed=True)
        statistics = grouped.agg(
            bar_count=("date", "size"),
            valid_price_count=("valid_close", "count"),
            valid_obi_count=("obi5", "count"),
            first_price=("valid_close", "first"),
            last_price=("valid_close", "last"),
            mean_obi5=("obi5", "mean"),
            mean_abs_obi5=("abs_obi5", "mean"),
            mean_obi1=("obi1", "mean"),
            amount=("amount_increment", "sum"),
            valid_amount_count=("amount_increment", "count"),
        )
        return statistics.add_prefix(f"{window_name}_")


    def _phase_space_hysteresis(minute: pd.DataFrame) -> pd.DataFrame:
        """
        Measure the 14:30-15:00 pressure/price path orientation.

        q_t = OBI5_t - OBI5_start
        x_t = log(P_t / P_start)

        The signed open-path area is
            A = 0.5 * sum(q_t*x_{t+1} - q_{t+1}*x_t).

        A > 0 means pressure tends to lead price (continuation-like).
        We define exhaustion_hysteresis = -A_normalized, so a larger value means
        price tends to lead subsequently displayed pressure. No future bars or
        future returns are used.
        """
        keys = ["trade_date", "instrument"]
        late = minute.loc[
            minute["window"].isin(["formation", "tail"]),
            ["trade_date", "instrument", "date", "valid_close", "obi5"],
        ].dropna(subset=["valid_close", "obi5"])

        def one_path(group: pd.DataFrame) -> pd.Series:
            group = group.sort_values("date", kind="mergesort")
            price = group["valid_close"].to_numpy(dtype="float64")
            pressure = group["obi5"].to_numpy(dtype="float64")
            count = len(group)
            if count < 20 or not np.all(price > 0):
                return pd.Series({
                    "hysteresis_bar_count": count,
                    "hysteresis_area": np.nan,
                    "hysteresis_exhaustion": np.nan,
                })

            x = np.log(price / price[0])
            q = pressure - pressure[0]
            dx = np.diff(x)
            dq = np.diff(q)
            price_energy = float(np.sqrt(np.sum(dx * dx)))
            pressure_energy = float(np.sqrt(np.sum(dq * dq)))
            denominator = price_energy * pressure_energy
            if denominator <= EPS:
                normalized = np.nan
                area = np.nan
            else:
                area = 0.5 * float(
                    np.sum(q[:-1] * x[1:] - q[1:] * x[:-1])
                )
                normalized = -area / denominator

            return pd.Series({
                "hysteresis_bar_count": count,
                "hysteresis_area": area,
                "hysteresis_exhaustion": normalized,
            })

        if late.empty:
            return pd.DataFrame()
        return late.groupby(keys, sort=False, observed=True).apply(one_path)


    def calculate_daily_overextension(
        minute: pd.DataFrame,
    ) -> pd.DataFrame:
        """Calculate late displacement divided by same-day realized volatility."""
        keys = ["trade_date", "instrument"]
        grouped = minute.groupby(keys, sort=False, observed=True)
        daily = grouped.agg(
            day_bar_count=("date", "size"),
            realized_variance=("return_squared", "sum"),
            day_amount=("amount_increment", "sum"),
            day_valid_amount_count=("amount_increment", "count"),
            amount_reset_count=("amount_reset", "sum"),
            amount_anomaly_count=("amount_anomaly", "sum"),
        )
        formation = _window_statistics(minute, "formation")
        tail = _window_statistics(minute, "tail")
        hysteresis = _phase_space_hysteresis(minute)
        daily = (
            daily.join(formation, how="left")
            .join(tail, how="left")
            .join(hysteresis, how="left")
        )

        daily["late_start_price"] = daily["formation_first_price"]
        daily["late_end_price"] = daily["tail_last_price"]
        daily["late_move"] = np.log(
            daily["late_end_price"] / daily["late_start_price"]
        )
        daily["intraday_volatility"] = np.sqrt(
            daily["realized_variance"]
        )
        daily["overextension"] = (
            daily["late_move"] / daily["intraday_volatility"]
        ).where(daily["intraday_volatility"] > EPS).clip(-5, 5)
        return daily.reset_index()


    def calculate_closing_crowding(
        daily: pd.DataFrame,
        mode: str,
    ) -> pd.DataFrame:
        """Replace OBIPersistence with a positive hysteresis exhaustion gate."""
        result = daily
        tail_mean_obi5 = result["tail_mean_obi5"]
        tail_mean_abs_obi5 = result["tail_mean_abs_obi5"]

        result["obi_persistence"] = (
            tail_mean_obi5.abs() / tail_mean_abs_obi5
        ).where(tail_mean_abs_obi5 > MIN_DEPTH_TOTAL).clip(0, 1)

        result["depth_coherence"] = (
            1.0
            - (result["tail_mean_obi1"] - tail_mean_obi5).abs() / 2.0
        ).where(
            result["tail_mean_obi1"].notna()
            & tail_mean_obi5.notna()
        ).clip(0, 1)

        result = _cross_sectional_robust_z(
            result,
            "hysteresis_exhaustion",
            "hysteresis_z",
        )
        clipped_hysteresis = result["hysteresis_z"].clip(
            -GATE_SCORE_CLIP,
            GATE_SCORE_CLIP,
        )
        result["hysteresis_gate"] = (
            0.25
            + 0.75 / (1.0 + np.exp(-clipped_hysteresis))
        ).where(result["hysteresis_z"].notna())

        # H1 is a clean replacement experiment, not an additional gate stacked
        # outside the already effective CHRF structure.
        persistence = result["hysteresis_gate"]
        coherence = (
            1.0
            if mode == "drop_depth_coherence"
            else result["depth_coherence"]
        )
        result["closing_crowding"] = (
            np.sign(result["late_move"])
            * tail_mean_obi5
            * persistence
            * coherence
        )
        return result


    def calculate_activity_crowding(
        daily: pd.DataFrame,
    ) -> pd.DataFrame:
        """Use amount, matching the validated version; never sum cumulative amount."""
        result = daily
        late_amount = (
            result["formation_amount"] + result["tail_amount"]
        )
        result["late_activity_share"] = (
            late_amount / result["day_amount"]
        ).where(result["day_amount"] > EPS).clip(0, 1)
        result["activity_crowding"] = (
            result["late_activity_share"] - EXPECTED_LATE_SHARE
        )
        return result


    def calculate_confidence_gate(
        daily: pd.DataFrame,
        mode: str,
    ) -> pd.DataFrame:
        """Create a strictly positive gate that cannot reverse Direction."""
        result = daily
        result = _cross_sectional_robust_z(
            result, "overextension", "overextension_z"
        )
        result = _cross_sectional_robust_z(
            result, "closing_crowding", "closing_crowding_z"
        )
        result = _cross_sectional_robust_z(
            result, "activity_crowding", "activity_crowding_z"
        )
        result["direction"] = -result["overextension_z"]

        if mode == "drop_activity_crowding":
            result["exhaustion_state"] = result["closing_crowding_z"]
        else:
            result["exhaustion_state"] = (
                WEIGHT_CLOSING * result["closing_crowding_z"]
                - WEIGHT_ACTIVITY * result["activity_crowding_z"]
            )

        score = result["exhaustion_state"].clip(
            -GATE_SCORE_CLIP, GATE_SCORE_CLIP
        )
        result["confidence_gate"] = (
            0.10 + 0.90 / (1.0 + np.exp(-score))
        )
        return result


    def calculate_final_factor(
        daily: pd.DataFrame,
        mode: str,
    ) -> pd.DataFrame:
        """Apply eligibility checks and final same-day robust normalization."""
        result = daily
        sufficient_bars = (
            (result["day_bar_count"] >= MIN_DAY_BARS)
            & (result["formation_bar_count"] >= MIN_FORMATION_BARS)
            & (result["tail_bar_count"] >= MIN_TAIL_BARS)
        )
        core_valid = result[
            [
                "direction",
                "closing_crowding_z",
                "activity_crowding_z",
                "confidence_gate",
            ]
        ].notna().all(axis=1)
        if mode == "drop_activity_crowding":
            core_valid = result[
                ["direction", "closing_crowding_z", "confidence_gate"]
            ].notna().all(axis=1)

        eligible = sufficient_bars & core_valid
        result["raw_factor"] = (
            result["direction"] * result["confidence_gate"]
        ).where(eligible)

        # The positive gate preserves Direction's sign before final centering.
        sign_violation = (
            result["raw_factor"].notna()
            & result["direction"].notna()
            & (
                np.sign(result["raw_factor"])
                != np.sign(result["direction"])
            )
        )
        if sign_violation.any():
            raise AssertionError("ConfidenceGate changed Direction sign")

        result = _cross_sectional_robust_z(
            result, "raw_factor", "factor"
        )
        return result


    def _aggregate_daily_factor(
        minute: pd.DataFrame,
        mode: str,
    ) -> pd.DataFrame:
        daily = calculate_daily_overextension(minute)
        daily = calculate_closing_crowding(daily, mode)
        daily = calculate_activity_crowding(daily)
        daily = calculate_confidence_gate(daily, mode)
        daily = calculate_final_factor(daily, mode)
        return daily


    # ---------------------------------------------------------------------------
    # Output validation and diagnostics
    # ---------------------------------------------------------------------------

    def _daily_diagnostics(daily: pd.DataFrame) -> pd.DataFrame:
        candidates = daily.groupby("trade_date", observed=True).size()
        valid = daily.groupby("trade_date", observed=True)["factor"].count()
        diagnostics = pd.DataFrame({
            "candidate_instruments": candidates,
            "valid_factors": valid,
        })
        diagnostics["coverage"] = (
            diagnostics["valid_factors"]
            / diagnostics["candidate_instruments"].clip(lower=1)
        )

        factor_stats = daily.groupby(
            "trade_date", observed=True
        )["factor"].agg(["mean", "std", "min", "max"])
        diagnostics = diagnostics.join(
            factor_stats.add_prefix("factor_"), how="left"
        )
        diagnostics["min_formation_bars"] = daily.groupby(
            "trade_date", observed=True
        )["formation_bar_count"].min()
        diagnostics["min_tail_bars"] = daily.groupby(
            "trade_date", observed=True
        )["tail_bar_count"].min()
        diagnostics["amount_resets"] = daily.groupby(
            "trade_date", observed=True
        )["amount_reset_count"].sum()
        diagnostics["amount_anomalies"] = daily.groupby(
            "trade_date", observed=True
        )["amount_anomaly_count"].sum()
        return diagnostics.reset_index().rename(
            columns={"trade_date": "date"}
        )


    def validate_factor_output(
        result: pd.DataFrame,
        audit: pd.DataFrame,
    ) -> None:
        assert list(result.columns) == ["date", "instrument", "factor"]
        assert not result.duplicated(["date", "instrument"]).any()
        assert np.isfinite(result["factor"]).all()
        if result.empty:
            raise ValueError("No valid factors were produced")

        low_coverage = audit.loc[
            audit["coverage"] < LOW_COVERAGE_WARNING
        ]
        if not low_coverage.empty:
            warnings.warn(
                f"Coverage below {LOW_COVERAGE_WARNING:.0%} on "
                f"{len(low_coverage)} trading days"
            )

        daily_mean = audit["factor_mean"].dropna()
        daily_std = audit["factor_std"].dropna()
        print("========== CHRF output audit ==========")
        print(
            f"date range: {result['date'].min().date()} -> "
            f"{result['date'].max().date()}"
        )
        print(
            f"trading days={result['date'].nunique():,}, "
            f"instruments={result['instrument'].nunique():,}, "
            f"rows={len(result):,}"
        )
        print(
            "daily coverage: "
            f"mean={audit['coverage'].mean():.2%}, "
            f"min={audit['coverage'].min():.2%}"
        )
        print(
            "factor: "
            f"mean={result['factor'].mean():.6f}, "
            f"std={result['factor'].std(ddof=1):.6f}, "
            f"min={result['factor'].min():.4f}, "
            f"max={result['factor'].max():.4f}"
        )
        print(
            "daily cross-sectional mean distribution: "
            f"mean={daily_mean.mean():.6f}, "
            f"std={daily_mean.std(ddof=1):.6f}, "
            f"min={daily_mean.min():.6f}, "
            f"max={daily_mean.max():.6f}"
        )
        print(
            "daily cross-sectional std distribution: "
            f"mean={daily_std.mean():.6f}, "
            f"std={daily_std.std(ddof=1):.6f}, "
            f"min={daily_std.min():.6f}, "
            f"max={daily_std.max():.6f}"
        )


    def describe_microstructure_audit(
        daily_detail: pd.DataFrame,
    ) -> Dict[str, pd.DataFrame]:
        """Optional research diagnostics; does not alter factor output."""
        quantiles = [0.01, 0.05, 0.50, 0.95, 0.99]

        def describe_column(
            frame: pd.DataFrame,
            column: str,
        ) -> pd.DataFrame:
            values = frame[column].replace([np.inf, -np.inf], np.nan)
            description = values.describe(percentiles=quantiles)
            return description.to_frame(column)

        up = daily_detail.loc[daily_detail["late_move"] > 0]
        down = daily_detail.loc[daily_detail["late_move"] < 0]
        return {
            "obi_persistence_all": describe_column(
                daily_detail, "obi_persistence"
            ),
            "obi_persistence_up": describe_column(up, "obi_persistence"),
            "obi_persistence_down": describe_column(
                down, "obi_persistence"
            ),
            "depth_coherence": describe_column(
                daily_detail, "depth_coherence"
            ),
            "confidence_gate": describe_column(
                daily_detail, "confidence_gate"
            ),
            "window_bar_counts": daily_detail[
                [
                    "trade_date",
                    "instrument",
                    "formation_bar_count",
                    "tail_bar_count",
                ]
            ].copy(),
        }


    # ---------------------------------------------------------------------------
    # Required competition interface
    # ---------------------------------------------------------------------------

    def _impl(
        datasources: Dict[str, str],
        start_datetime: object,
        end_datetime: object,
    ) -> pd.DataFrame:
        """Return exactly date, instrument and factor."""
        if ABLATION_MODE not in VALID_ABLATION_MODES:
            raise ValueError(f"Invalid ABLATION_MODE: {ABLATION_MODE!r}")
        if not isinstance(datasources, dict):
            raise TypeError("datasources must be a dict")
        if "bar1m" not in datasources:
            raise KeyError("datasources must contain bar1m")

        start = _as_timestamp(start_datetime, "start_datetime")
        end = _as_timestamp(end_datetime, "end_datetime")
        if start > end:
            raise ValueError("start_datetime must not exceed end_datetime")

        # Inclusive trading-date bounds plus causal warmup for the temporal state.
        evaluation_start = start.normalize()
        evaluation_end = end.normalize()
        query_start = evaluation_start - pd.Timedelta(days=WARMUP_CALENDAR_DAYS)
        query_end = evaluation_end + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)

        bar_source = str(datasources["bar1m"])
        daily_parts: List[pd.DataFrame] = []

        for chunk_start, chunk_end in _date_chunks(query_start, query_end):
            raw = _query_minute_chunk(bar_source, chunk_start, chunk_end)
            if raw.empty:
                continue

            minute = prepare_minute_data(raw)
            del raw
            minute = convert_cumulative_fields(minute)
            minute = calculate_minute_returns(minute)
            minute = calculate_obi_features(minute)

            daily = _aggregate_daily_factor(minute, ABLATION_MODE)
            if not daily.empty:
                daily_parts.append(daily)

            del minute, daily
            gc.collect()

        if not daily_parts:
            raise ValueError("No minute data found in the requested interval")

        daily_all = pd.concat(daily_parts, ignore_index=True)
        del daily_parts
        daily_all["trade_date"] = pd.to_datetime(
            daily_all["trade_date"], errors="coerce"
        ).dt.normalize()
        daily_all["instrument"] = daily_all["instrument"].astype("string")
        daily_all = daily_all.sort_values(
            ["trade_date", "instrument"], kind="mergesort"
        ).drop_duplicates(
            ["trade_date", "instrument"], keep="last"
        ).reset_index(drop=True)

        # Causal L1 friction / stick-slip update. It is the exact proximal
        # solution of 0.5*(x-signal)^2 + lambda*abs(x-previous_state).
        # Small day-to-day changes are held; large innovations pass through after
        # paying the fixed friction threshold. No future observation is used.
        daily_all = daily_all.sort_values(
            ["instrument", "trade_date"], kind="mergesort"
        ).reset_index(drop=True)

        def apply_causal_friction(series: pd.Series) -> pd.Series:
            values = pd.to_numeric(series, errors="coerce").to_numpy(
                dtype="float64", copy=False
            )
            stable = np.full(len(values), np.nan, dtype="float64")
            previous = np.nan
            for position, current in enumerate(values):
                if not np.isfinite(current):
                    continue
                if not np.isfinite(previous):
                    previous = current
                else:
                    difference = current - previous
                    movement = max(abs(difference) - FRICTION_LAMBDA, 0.0)
                    if movement > 0.0:
                        previous = previous + np.sign(difference) * movement
                stable[position] = previous
            return pd.Series(stable, index=series.index, dtype="float64")

        daily_all["pre_friction_factor"] = daily_all["factor"].astype("float64")
        daily_all["friction_state"] = daily_all.groupby(
            "instrument", sort=False, observed=True
        )["pre_friction_factor"].transform(apply_causal_friction)
        daily_all = _cross_sectional_robust_z(
            daily_all, "friction_state", "factor"
        )

        # Remove warmup observations only after the causal state has been built.
        daily_all = daily_all.loc[
            daily_all["trade_date"].between(query_start, evaluation_end)
        ].copy()
        daily_all = daily_all.sort_values(
            ["trade_date", "instrument"], kind="mergesort"
        ).reset_index(drop=True)

        _LAST_CHRF_AUDIT = _daily_diagnostics(daily_all)

        result = daily_all.loc[
            np.isfinite(daily_all["factor"]),
            ["trade_date", "instrument", "factor"],
        ].rename(columns={"trade_date": "date"})
        result["date"] = pd.to_datetime(
            result["date"], errors="coerce"
        ).dt.normalize()
        result["instrument"] = result["instrument"].astype("string")
        result["factor"] = pd.to_numeric(
            result["factor"], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

        # Strict official-calendar alignment (same protection as CHRF V4).
        pool_sql = """
            SELECT date, instrument
            FROM bigalpha_2026_instruments
            ORDER BY date, instrument
        """
        pool = _as_frame(
            dai.query(
                pool_sql,
                filters={
                    "date": [
                        query_start.strftime("%Y-%m-%d 00:00:00"),
                        evaluation_end.strftime("%Y-%m-%d 23:59:59"),
                    ]
                },
                compression=True,
            )
        )
        if pool.empty:
            raise ValueError("Official instrument calendar query returned no rows")
        missing_pool_columns = {"date", "instrument"} - set(pool.columns)
        if missing_pool_columns:
            raise KeyError(
                "bigalpha_2026_instruments is missing columns: "
                f"{sorted(missing_pool_columns)}"
            )

        pool = pool.loc[:, ["date", "instrument"]].copy()
        pool["date"] = pd.to_datetime(
            pool["date"], errors="coerce"
        ).dt.normalize()
        pool["instrument"] = pool["instrument"].astype("string")
        pool = pool.dropna(subset=["date", "instrument"])
        pool = pool.loc[
            pool["date"].between(query_start, evaluation_end)
        ]
        pool = pool.drop_duplicates(
            ["date", "instrument"], keep="last"
        ).sort_values(
            ["date", "instrument"], kind="mergesort"
        ).reset_index(drop=True)
        if pool.empty:
            raise ValueError("Official instrument calendar has no evaluation rows")

        # V6 compliance: expected trading dates must not be derived only from
        # the same constituent table used to create the output.  Otherwise a
        # missing date in that table disappears from both output and audit.
        minute_calendar = pd.Index(
            daily_all.loc[
                daily_all["trade_date"].between(
                    evaluation_start, evaluation_end
                ),
                "trade_date",
            ].dropna().drop_duplicates().sort_values()
        )
        pool_calendar = pd.Index(
            pool.loc[
                pool["date"].between(evaluation_start, evaluation_end),
                "date",
            ].dropna().drop_duplicates().sort_values()
        )
        expected_dates = minute_calendar.union(pool_calendar).sort_values()
        if len(expected_dates) == 0:
            raise ValueError("No evaluation trading dates found")

        # Complete a pool date only from the most recent composition known on
        # or before that date.  This handles a one-day auxiliary-table lag
        # without using future constituents.
        known_pool_dates = pd.Index(
            pool["date"].drop_duplicates().sort_values()
        )
        completed_pool_parts = []
        carried_pool_dates = []
        for trading_date in expected_dates:
            same_day = pool.loc[
                pool["date"].eq(trading_date), ["date", "instrument"]
            ]
            if not same_day.empty:
                completed_pool_parts.append(same_day)
                continue

            prior_dates = known_pool_dates[known_pool_dates < trading_date]
            if len(prior_dates) == 0:
                raise ValueError(
                    "No causal constituent history for trading date "
                    f"{trading_date:%Y-%m-%d}"
                )
            prior_date = prior_dates[-1]
            carried_pool = pool.loc[
                pool["date"].eq(prior_date), ["date", "instrument"]
            ].copy()
            carried_pool["date"] = trading_date
            completed_pool_parts.append(carried_pool)
            carried_pool_dates.append(trading_date)

        pool_evaluation = pd.concat(
            completed_pool_parts, ignore_index=True
        ).drop_duplicates(
            ["date", "instrument"], keep="last"
        ).sort_values(
            ["date", "instrument"], kind="mergesort"
        ).reset_index(drop=True)

        # Warmup rows are retained only to support causal factor carry.
        pool_warmup = pool.loc[pool["date"] < evaluation_start].copy()
        pool = pd.concat(
            [pool_warmup, pool_evaluation], ignore_index=True
        ).drop_duplicates(
            ["date", "instrument"], keep="last"
        ).sort_values(
            ["date", "instrument"], kind="mergesort"
        ).reset_index(drop=True)

        result = pool.merge(
            result,
            on=["date", "instrument"],
            how="left",
            validate="one_to_one",
        )
        # Never fill an entire missing day with a constant: platform
        # standardization would turn a zero-variance day back into all missing.
        # Causally carry the latest valid friction state from warmup history.
        result = result.sort_values(
            ["instrument", "date"], kind="mergesort"
        ).reset_index(drop=True)
        missing_before_carry = result["factor"].isna()
        result["factor"] = result.groupby(
            "instrument", sort=False, observed=True
        )["factor"].ffill()
        carried_values = int(
            (missing_before_carry & result["factor"].notna()).sum()
        )
        remaining_before_neutral = result["factor"].isna()
        result["factor"] = result["factor"].fillna(0.0).astype("float64")

        result["factor"] = result.groupby(
            "date", group_keys=False, observed=True
        )["factor"].transform(robust_zscore)
        result = result.loc[
            result["date"].between(evaluation_start, evaluation_end)
        ].copy()
        result = result.sort_values(
            ["date", "instrument"], kind="mergesort"
        ).reset_index(drop=True)

        returned_dates = pd.Index(
            result.loc[np.isfinite(result["factor"]), "date"]
            .drop_duplicates().sort_values()
        )
        absent_dates = expected_dates.difference(returned_dates)
        if len(absent_dates) > 0:
            raise ValueError(
                "Strict calendar alignment has missing dates: "
                f"{absent_dates.strftime('%Y-%m-%d').tolist()}"
            )

        daily_quality = result.groupby(
            "date", observed=True
        )["factor"].agg(
            finite_count=lambda values: np.isfinite(values).sum(),
            cross_section_std="std",
            unique_values="nunique",
        )
        bad_daily = daily_quality.loc[
            (daily_quality["finite_count"] <= 0)
            | ~np.isfinite(daily_quality["cross_section_std"])
            | (daily_quality["cross_section_std"] <= EPS)
            | (daily_quality["unique_values"] <= 1)
        ]
        if not bad_daily.empty:
            raise ValueError(
                "H2 daily factor lacks usable cross-sectional dispersion: "
                f"{bad_daily.index.strftime('%Y-%m-%d').tolist()}"
            )

        print(
            "H2 causal friction completion: "
            f"lambda={FRICTION_LAMBDA:.2f}, "
            f"warmup_days={WARMUP_CALENDAR_DAYS}, "
            f"dates={len(expected_dates)}, "
            f"carried_values={carried_values:,}, "
            f"neutral_new_stock_values={int(remaining_before_neutral.sum()):,}, "
            f"carried_pool_dates={len(carried_pool_dates):,}, "
            f"min_daily_std={daily_quality['cross_section_std'].min():.6f}"
        )
        validate_factor_output(result, _LAST_CHRF_AUDIT)
        print(f"active mode: {ABLATION_MODE}")
        return result[["date", "instrument", "factor"]]


    # ---------------------------------------------------------------------------
    # Lightweight sign/unit tests (run once after defining Code Block 1)
    # ---------------------------------------------------------------------------

    def run_chrf_unit_tests() -> None:
        """Verify both up/down crowding signs and positive-gate invariants."""
        synthetic = pd.DataFrame({
            "trade_date": pd.to_datetime(["2023-01-03"] * 4),
            "late_move": [0.01, -0.01, 0.01, -0.01],
            "tail_mean_obi5": [0.40, -0.40, -0.40, 0.40],
            "tail_mean_abs_obi5": [0.50, 0.50, 0.50, 0.50],
            "tail_mean_obi1": [0.35, -0.35, -0.35, 0.35],
            "hysteresis_exhaustion": [-1.0, 1.0, -0.5, 0.5],
        })
        tested = calculate_closing_crowding(synthetic, "main")
        assert tested.loc[0, "closing_crowding"] > 0
        assert tested.loc[1, "closing_crowding"] > 0
        assert tested.loc[2, "closing_crowding"] < 0
        assert tested.loc[3, "closing_crowding"] < 0

        scores = pd.Series([-100.0, -6.0, 0.0, 6.0, 100.0])
        gates = 0.10 + 0.90 / (
            1.0 + np.exp(-scores.clip(-6.0, 6.0))
        )
        assert gates.between(0.10, 1.00, inclusive="both").all()
        directions = pd.Series([-2.0, -1.0, 0.0, 1.0, 2.0])
        assert (
            np.sign(directions * gates) == np.sign(directions)
        ).all()
        print("CHRF unit tests passed.")
    return _impl(datasources, start_date, end_date)
